# Module 7: Seasonal Adjustment

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 18](../Beginner/Topic_18_Year_Over_Year_Comparison.md) removed
the season by comparing July to July. That works, and it costs you eleven
twelfths of your data: only one comparison a year.

Seasonal adjustment does the same job differently. It divides out the seasonal
pattern so that **every consecutive month becomes comparable**, which is what
you need for a monthly report.

**About 15 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

## 2. Estimate the seasonal factors, then divide them out

The factors come from [Module 5](Module_05_Decomposition.md). Adjustment is one
division.

    adjusted = observed / seasonal factor

A month with a factor of 1.45 is one where the calendar alone would put the
count 45 percent above the level, so dividing by 1.45 asks what that month
would have been on an ordinary calendar.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(np.log(grandview), period=12, robust=True).fit()
factor = np.exp(stl.seasonal)
adjusted = grandview / factor

names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
idx = factor.groupby(factor.index.month).mean().round(2)
idx.index = names
print(idx.to_string())

## 3. What adjustment does and does not do

In [ ]:
table = pd.DataFrame({"as reported": grandview, "seasonally adjusted": adjusted.round(1),
                      "factor": factor.round(2)}).loc["2023"]
table.index = names
table

January's 50 becomes 61, because January is a naturally quiet month and 50 was
low even for a January. July's 157 becomes 107, because a large part of that
157 was the calendar.

In [ ]:
y23 = adjusted.loc["2023"]
r23 = grandview.loc["2023"]
print(f"as reported      : {r23.min():.0f} to {r23.max():.0f}, a range of {r23.max()-r23.min():.0f}")
print(f"adjusted         : {y23.min():.0f} to {y23.max():.0f}, a range of {y23.max()-y23.min():.0f}")

mom_raw = 100 * (grandview / grandview.shift(1) - 1)
mom_adj = 100 * (adjusted / adjusted.shift(1) - 1)
print(f"\nmonth on month change, standard deviation across all months")
print(f"  as reported: {mom_raw.std():.1f} percent")
print(f"  adjusted   : {mom_adj.std():.1f} percent")

The 2023 range halves and the month on month variability drops by about a
third. What remains is real movement, and there is still plenty of it.

**Adjustment removes the calendar, not the noise.** A month can still be high
for no reason at all. That is [Module 8](Module_08_Rolling_Statistics_And_Control_Limits.md).

In [ ]:
print(f"June 2023 against May: as reported {mom_raw.loc['2023-06-01']:+.1f} percent, "
      f"adjusted {mom_adj.loc['2023-06-01']:+.1f} percent")
print(f"February 2023 against January: as reported {mom_raw.loc['2023-02-01']:+.1f} percent, "
      f"adjusted {mom_adj.loc['2023-02-01']:+.1f} percent")

The February line is the instructive one. Adjustment made the rise look
**larger**, not smaller, because January 2023 was unusually low even after
allowing for January being a quiet month. Adjustment is not a smoothing
device that shrinks everything. It removes one specific, predictable component,
and whatever it uncovers is what was underneath.

## 4. Every agency needs its own factors

Pinecrest State University serves a campus. Applying the statewide seasonal
shape to it would get the direction of the correction backwards for half the
year.

In [ ]:
def factors(s):
    return np.exp(STL(np.log(s), period=12, robust=True).fit().seasonal) \
           .groupby(s.index.month).mean()

both = pd.DataFrame({"Grandview": factors(grandview),
                     "Pinecrest State University": factors(series("A010"))}).round(2)
both.index = names
both

Grandview's quietest month is February; Pinecrest's is June, when the students
leave. A single statewide factor would inflate Pinecrest's summer and deflate
its autumn, manufacturing a pattern that is not there.

## 5. When you may write "seasonally adjusted"

| Condition | Why |
|---|---|
| At least three full years, preferably five | the factor for each month is an average across years |
| Factors estimated from this agency | seasonality differs by agency type |
| The estimation window stated | factors change if you re estimate on different years |
| Provisional months excluded | they drag the recent factors |
| The unadjusted series shown too | readers must be able to see what was done |

And one honest caveat. The factors at the **end** of the series are the least
certain, because loess has data on only one side of them. The most recent
adjusted point is the one most likely to be revised when next month arrives.

In [ ]:
short = grandview.loc[:"2024-12"]
f_short = np.exp(STL(np.log(short), period=12, robust=True).fit().seasonal)

both_runs = pd.DataFrame({
    "using data to 2024": f_short.loc["2024"].round(3).values,
    "using data to 2026": factor.loc["2024"].round(3).values}, index=names)
both_runs["revision"] = (both_runs.iloc[:, 1] - both_runs.iloc[:, 0]).round(3)
both_runs

The 2024 factors changed once 2025 and 2026 arrived, and the largest revisions
are in the later months of 2024, nearest the end of the shorter series. An
adjusted figure published in January 2025 was provisional in a second sense,
beyond the reporting lag: the adjustment itself was not final.

## 6. What to carry away

- Adjustment makes **consecutive months** comparable, which year over year cannot.
- It removes the calendar and leaves the noise and the trend.
- Factors belong to the agency, the period and the series they were estimated on.
- Publish the unadjusted series alongside, always.
- Expect the most recent adjusted points to be revised.

## Exercise

Seasonally adjust Cedar Falls and check what happens to June 2021, the month of
civil unrest.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    s = series(AGENCY)
    f = np.exp(STL(np.log(s), period=12, robust=True).fit().seasonal)
    a = s / f
    out = pd.DataFrame({"as reported": s, "factor": f.round(2),
                        "adjusted": a.round(1)}).loc["2021-04":"2021-08"]
    print(out.to_string())
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

June 2021 stays enormous. The June factor is about 1.1, so adjustment barely
touches it, and 176 becomes about 161.

That is the correct behaviour and it is worth stating plainly: **seasonal
adjustment is not outlier removal.** A week of civil unrest is not a calendar
effect, so nothing in the adjustment should absorb it. The month remains
visible, which is exactly what you want, and it is the subject of the next
module.

</details>

---

**Next:** [Module 8, Rolling Statistics and Control Limits](Module_08_Rolling_Statistics_And_Control_Limits.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*